# 🫁 CheXpert Medical AI - Tự động Huấn luyện & Backup trên Google Colab GPU

Notebook này được tích hợp đầy đủ:
1. **Hiển thị tiến trình trực tiếp**: Xem thanh tiến trình (progress bar), Loss từng batch, Learning rate và bảng điểm AUC từng bệnh lý sau mỗi epoch.
2. **Tự động Backup lên Google Drive**: Sau mỗi epoch, model và điểm số được sao lưu ngay vào Google Drive cá nhân của bạn.
3. **Chạy tiếp không cần train lại từ đầu (Resume Training)**: Nếu Colab bị mất mạng hay ngắt kết nối giữa chừng, bạn chỉ cần chạy cell tiếp theo là model sẽ tự động load checkpoint cũ và tiếp tục train!

## 1. Kết nối Google Drive để tự động lưu Backup

In [ ]:
from google.colab import drive
import os

# Gắn Google Drive để sao lưu model an toàn
drive.mount('/content/drive')
backup_dir = '/content/drive/MyDrive/chex_backup'
os.makedirs(backup_dir, exist_ok=True)
print(f"Thư mục sao lưu trên Google Drive đã sẵn sàng: {backup_dir}")

## 2. Kiểm tra GPU & Clone dự án

In [ ]:
!nvidia-smi
!git clone https://github.com/qdat2644/chex.git
%cd chex
!pip install -q -r requirements.txt
!pip install -q kaggle

## 3. Tự động xác thực Kaggle Token & Tải Dataset

In [ ]:
import os
import json

# Cấu hình tự động Kaggle Token
KAGGLE_TOKEN = "KGAT_66e946b810c28ffaeb913743194bd633"
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN
os.environ["KAGGLE_KEY"] = KAGGLE_TOKEN
os.environ["KAGGLE_USERNAME"] = "qdat2644"

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_path, "w") as f:
    json.dump({"username": "qdat2644", "key": KAGGLE_TOKEN, "token": KAGGLE_TOKEN}, f)
os.chmod(kaggle_path, 0o600)

# Tải và giải nén (nếu chưa có)
if not os.path.exists('archive/train.csv'):
    print("Đang tải dataset CheXpert từ Kaggle...")
    !mkdir -p archive
    !kaggle datasets download -d ashery/chexpert -p archive/ --unzip
    print("Tải và giải nén xong!")
else:
    print("Dataset đã có sẵn trong archive/")

## 4. Huấn luyện Mô hình Mới (Kèm Live Progress & Auto-Backup lên Drive)

Chọn kiến trúc: `convnext_small` (mạnh nhất) hoặc `densenet121`

In [ ]:
!python scripts/train.py \
    --data-root archive \
    --arch convnext_small \
    --loss asl \
    --image-size 224 \
    --epochs 6 \
    --batch-size 32 \
    --lr 1e-4 \
    --uncertain-policy u_ones_zeros \
    --scheduler cosine \
    --pretrained \
    --amp \
    --backup-dir /content/drive/MyDrive/chex_backup \
    --output checkpoints/chexpert_convnext_small.pt

## 4b. [Tùy chọn] Chạy tiếp từ Checkpoint đã lưu trên Drive (Nếu bị rớt mạng)

Nếu phiên Colab trước bị ngắt kết nối giữa chừng, bạn chỉ cần chạy ô này để tiếp tục train từ epoch đang dở:

In [ ]:
# Tự động tìm checkpoint gần nhất trên Drive để chạy tiếp
resume_ckpt = '/content/drive/MyDrive/chex_backup/chexpert_convnext_small_last.pt'
if os.path.exists(resume_ckpt):
    print(f"Tìm thấy checkpoint sao lưu: {resume_ckpt}. Bắt đầu chạy tiếp...")
    !python scripts/train.py \
        --data-root archive \
        --arch convnext_small \
        --loss asl \
        --image-size 224 \
        --epochs 6 \
        --batch-size 32 \
        --lr 1e-4 \
        --uncertain-policy u_ones_zeros \
        --scheduler cosine \
        --amp \
        --backup-dir /content/drive/MyDrive/chex_backup \
        --resume /content/drive/MyDrive/chex_backup/chexpert_convnext_small_last.pt \
        --output checkpoints/chexpert_convnext_small.pt
else:
    print("Chưa có checkpoint backup để resume.")

## 5. Đánh giá & Tự động Hiệu chuẩn Ngưỡng phân loại F1 (Thresholds)

In [ ]:
# Đánh giá và lưu ngưỡng tối ưu
!python scripts/evaluate.py \
    --data-root archive \
    --checkpoint checkpoints/chexpert_convnext_small.pt \
    --output-thresholds outputs/evaluation/thresholds.json

# Sao lưu luôn file thresholds vào Google Drive
!cp outputs/evaluation/thresholds.json /content/drive/MyDrive/chex_backup/

## 6. Tải Model & File Thresholds về máy tính

In [ ]:
from google.colab import files

print("Tải file model weights (.pt)...")
files.download('checkpoints/chexpert_convnext_small.pt')

print("Tải file thresholds.json...")
files.download('outputs/evaluation/thresholds.json')

print("Hoàn tất! Cả 2 file cũng đã được lưu an toàn vĩnh viễn trên Google Drive (thư mục: chex_backup)!")